# 2일차 2교시 — Double DQN 소개

**PyTorch로 배우는 강화학습 · 2일차 Value-based & Policy-based Methods · 2026-07-28 (화)**

이애본 (Ph.D Aebon) · DreamIT Biz · https://pytorch26.dreamitbiz.com

---

## 🎯 학습목표

- Q-Learning의 최대화 편향(maximization bias)이 왜 생기는지 이해한다
- Double DQN이 선택과 평가를 분리하는 방식을 설명할 수 있다

---

# ⚡ 실행 방법 두 가지 — 편한 쪽을 고르세요

### 방법 ① 통째로 한 번에
바로 아래 **[통째로 실행]** 셀 **하나만** 실행하면 끝까지 돕니다.
결과부터 보고 싶으신 분께 권합니다.

### 방법 ② 단계별로 하나씩
그 아래 **[단계별]** 부분을 위에서부터 `Shift + Enter` 로 하나씩 실행하세요.
모두 **7칸**입니다. 한 칸 돌리고 결과 보고 넘어가면 됩니다.

> **이 교시는 혼자 돌아갑니다.** 앞 교시를 먼저 실행하지 않아도 됩니다.
> (앞 교시에서 만든 것을 이 노트북 안에 다시 넣어 뒀습니다 — 사이트의 *이 교시 전체 코드* 와 같은 판입니다.)
> 설치할 것도 없습니다 — 코랩에 다 들어 있습니다.

---

# ① 통째로 한 번에 실행

GitHub 에서 원본을 받아 그대로 돌립니다. 원본이 고쳐지면 자동으로 최신을 받습니다.

In [ ]:
!curl -sL https://raw.githubusercontent.com/aebonlee/pytorch26-lab/main/day2/standalone/02_ddqn_target.py -o 02_ddqn_target.py
!python 02_ddqn_target.py

---

# ② 단계별로 하나씩 실행

이 교시 내용이 **7칸**입니다.
위에서부터 `Shift + Enter`.

> ①을 이미 돌리셨어도 상관없습니다. 처음부터 다시 하는 것과 같습니다.

### 1 / 7 칸

In [ ]:
# ============================================================
# 2일차 2교시 — Double DQN 소개
# 복사해서 그대로 실행하면 됩니다. 고칠 것 없습니다.
# ------------------------------------------------------------
# 이 교시 코드는 앞 교시의 변수·클래스를 이어 씁니다.
# 그래서 이 블록에는 **여기까지 필요한 코드가 전부** 들어 있습니다.
# (수업용 코드만 따로 복사하면 NameError 가 납니다 — 그건 정상입니다.)
# ============================================================

# ── 1교시에서 이어받음 — DQN 소개 ──
import random
from collections import deque
import numpy as np
import torch

### 2 / 7 칸

In [ ]:
class ReplayBuffer:
    """경험 재현 버퍼 — DQN, DDPG, SAC 3일 내내 재사용합니다"""
    def __init__(self, capacity=100_000, action_dtype=torch.int64):
        # action_dtype: 오늘 DQN은 행동이 "몇 번 행동"인 정수라 int64입니다.
        # 3일차 DDPG·SAC는 행동이 연속값(실수 벡터)이므로 float32로 바꿔 씁니다
        #   buffer = ReplayBuffer(100_000, action_dtype=torch.float32)
        # int64로 두면 실수 행동이 정수로 잘려 학습이 통째로 망가집니다.
        self.buffer = deque(maxlen=capacity)
        self.action_dtype = action_dtype

    def push(self, s, a, r, s_next, done):
        self.buffer.append((s, a, r, s_next, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        s, a, r, s_next, done = zip(*batch)
        # np.array로 한 번 묶고 텐서로 바꿉니다.
        # 배열 리스트를 텐서로 바로 만들면 파이토치가 하나씩 복사해 매우 느립니다.
        return (torch.as_tensor(np.array(s), dtype=torch.float32),
                torch.as_tensor(np.array(a), dtype=self.action_dtype),
                torch.as_tensor(np.array(r), dtype=torch.float32),
                torch.as_tensor(np.array(s_next), dtype=torch.float32),
                torch.as_tensor(np.array(done), dtype=torch.float32))

    def __len__(self):
        return len(self.buffer)

### 3 / 7 칸

In [ ]:
# ── 오늘 이 교시 — Double DQN 소개 ──
import torch

# DQN vs Double DQN — 목표(target) 계산의 차이

### 4 / 7 칸

In [ ]:
@torch.no_grad()

### 5 / 7 칸

In [ ]:
def dqn_target(q_target, r, s_next, done, gamma=0.99):
    max_q = q_target(s_next).max(dim=1).values          # 타깃넷이 선택+평가
    return r + gamma * max_q * (1 - done)

### 6 / 7 칸

In [ ]:
@torch.no_grad()

### 7 / 7 칸

In [ ]:
def double_dqn_target(q_online, q_target, r, s_next, done, gamma=0.99):
    best_a = q_online(s_next).argmax(dim=1, keepdim=True)   # 선택: 온라인넷
    max_q = q_target(s_next).gather(1, best_a).squeeze(1)   # 평가: 타깃넷
    return r + gamma * max_q * (1 - done)

---

## 막히면

- 사이트의 같은 교시를 보세요 — 실행 결과와 해설이 그대로 있습니다.
  https://pytorch26.dreamitbiz.com/#/day/2/2
- 오류가 나면 **[막힐 때]** 메뉴부터.
  https://pytorch26.dreamitbiz.com/#/help

---

*Ph.D Aebon & Claude Code 협작 전자출판 도서 · © 2026 DreamIT Biz*